# Train the L2 fraud-detection model on Colab

Runs two training jobs against a GPU runtime and hands you back checkpoints in the same format `packages/models/train_baseline.py` and `packages/fl/server.py` produce locally, so they drop straight into `packages/models/checkpoints/` when you're done:

1. **Centralised baseline** on the real Elliptic dataset (165 anonymised features) — `baseline.pt`.
2. **Federated run** (FedProx, 3 simulated non-IID clients) on the L2 simulator's 4-feature schema (`value_in`, `value_out`, `degree_in`, `degree_out`) — `federated_fedprox.pt`. This one is what the app's *Test a Transaction* tab can score, since that feature only accepts the 4-feature simulator schema (Elliptic's features are anonymised, so there's no meaningful way to hand-build a transaction against a baseline checkpoint).

**Before you start:** Runtime → Change runtime type → GPU (T4 is enough).

**Security note:** do **not** pass `--on-chain` to either script in this notebook. That path needs `DEPLOYER_PRIVATE_KEY` / client keys from your local `.env` / `wallets.local.json`, and Colab notebooks (and their outputs) are easy to accidentally leave in Drive or share with secrets embedded. Do on-chain commitment runs locally only, straight from your machine.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repository onto this runtime

The repo is public at `github.com/nasredeenabdulhaleem/l2-fraud-fl`, so a plain clone works with no auth needed. This pulls whatever is on `main` at clone time — if you've made local commits since your last push, push them first so Colab sees them (uncommitted local changes never show up here regardless).

In [ ]:
!git clone https://github.com/nasredeenabdulhaleem/l2-fraud-fl.git
%cd l2-fraud-fl

## 2. Install dependencies

Colab already ships a CUDA-matched `torch` build — reinstalling `torch==2.2.2` from `requirements.txt` would fight that and likely leave you on CPU, so everything except the `torch` line installs as-is. `torch-geometric` itself is version-pinned but pure-Python for the ops this project uses (`SAGEConv`, `global_mean_pool`), so it installs cleanly against whatever torch Colab gives you.

In [ ]:
!grep -v '^torch==' requirements.txt > /tmp/requirements.colab.txt
!pip install -q -r /tmp/requirements.colab.txt
!pip install -q -e .

import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())

## 3. Centralised baseline on Elliptic

`EllipticLoader` downloads the raw CSVs via PyTorch Geometric on first use if they aren't already under `data/elliptic/raw/` — the feature file is ~690MB uncompressed, so the first run takes a few minutes.

In [ ]:
!python -m packages.models.train_baseline --source elliptic --epochs 30 --out packages/models/checkpoints/baseline.pt

## 4. Federated run (FedProx, 3 clients)

`packages/fl/server.py` and `packages/fl/client.py` talk over real sockets (`--server 127.0.0.1:8080`), not Flower's simulation API — there's no `start_simulation` entry point in this codebase. That still works fine in one Colab runtime: launch the server and three clients as background processes on localhost, exactly like running them in four terminals locally. `--no-telemetry` skips posting to the dashboard backend (nothing to post to from here); `--on-chain` is deliberately omitted per the note at the top.

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

LOG_DIR = Path("/content/fl_logs")
LOG_DIR.mkdir(exist_ok=True)
NUM_CLIENTS = 3

def spawn(name, args):
    log = open(LOG_DIR / f"{name}.log", "w")
    return subprocess.Popen([sys.executable, "-m", *args], stdout=log, stderr=subprocess.STDOUT)

server_proc = spawn("server", [
    "packages.fl.server", "--strategy", "fedprox", "--rounds", "10",
    "--min-clients", str(NUM_CLIENTS), "--no-telemetry",
])
time.sleep(5)  # give the server time to bind before clients try to connect

client_procs = [
    spawn(f"client_{i}", [
        "packages.fl.client", "--client-id", str(i), "--num-clients", str(NUM_CLIENTS),
        "--server", "127.0.0.1:8080",
    ])
    for i in range(NUM_CLIENTS)
]

print(f"server pid={server_proc.pid}, client pids={[p.pid for p in client_procs]}")
print(f"logs under {LOG_DIR}")

In [ ]:
TIMEOUT_S = 20 * 60
waited = 0
while any(p.poll() is None for p in client_procs) and waited < TIMEOUT_S:
    time.sleep(10)
    waited += 10

if waited >= TIMEOUT_S:
    print("timed out waiting for clients -- check the logs below before continuing")
else:
    print(f"all clients finished after ~{waited}s")

# The server saves its checkpoint right after the last round finalises, then exits.
server_proc.wait(timeout=60)
print("server exit code:", server_proc.returncode)

for log_file in sorted(LOG_DIR.glob("*.log")):
    print(f"\n----- {log_file.name} (last 15 lines) -----")
    print("\n".join(log_file.read_text().splitlines()[-15:]))

## 5. Download the checkpoints

Save these back into your local `packages/models/checkpoints/`. Any checkpoint with `in_dim == 4` (the federated one) shows up automatically in the app's *Test a Transaction* model picker the next time the backend starts; the Elliptic baseline (`in_dim == 165`) is listed too but marked not scorable there, since its features aren't something you can hand-type.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/checkpoints", "zip", "packages/models/checkpoints")
files.download(archive)